# 프로젝트 4 - Weekend 3: Multimodal Graph-RAG (Week 12 적용)

**목표** — Week 12에서 배운 **GraphRAG 패턴**을 Weekend 2의 **멀티모달 Document**에 적용합니다. PDF 표/이미지, 오디오, 비디오에서 *함께* 엔티티/관계를 추출해 단일 KG를 만들고, Leiden 커뮤니티 + 요약 + Local/Global Search까지 운영급으로 완성합니다.

**학습 목표**:
1. 멀티모달 Document → 청크별 LLM 엔티티/관계 추출 (`element_type` 메타 보존)
2. 청크 결과 병합 → `nx.DiGraph` (노드에 `source_modalities` 속성)
3. **Leiden 커뮤니티** 탐지 + modality 분포 분석
4. **LLM 커뮤니티 요약** (modality 정보 포함)
5. **Multimodal Local/Global Search** — 엔티티/요약 기반 + modality filter
6. **Multimodal Graph-RAG** — Week 12 패턴을 적용한 답변 생성
7. modality-aware 가중치 (표=정량, audio=발화, video=시각 등)
8. LLM-as-Judge로 Graph-RAG vs baseline 비교
9. `MultimodalGraphRAGService` 통합 클래스


In [ ]:
# 환경 설정 및 라이브러리 설치
!pip install -q langchain langchain-openai langchain-community faiss-cpu \
    pymupdf pillow pandas pydantic networkx graspologic matplotlib python-dotenv


In [ ]:
import os
import io
import json
import base64
from pathlib import Path
from collections import Counter, defaultdict
from dotenv import load_dotenv

load_dotenv()

import fitz
import pandas as pd
import networkx as nx
from PIL import Image
from pydantic import BaseModel, Field

from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.documents import Document
from langchain_core.messages import HumanMessage, SystemMessage

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
vision = ChatOpenAI(model="gpt-4o-mini", temperature=0)
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

DATA_DIR = Path("data")
PDF_REPORT = DATA_DIR / "quarterly_report.pdf"
IMG_DIR = DATA_DIR / "_extracted"
IMG_DIR.mkdir(exist_ok=True)
assert PDF_REPORT.exists(), f"❌ {PDF_REPORT} 없음"
print("✅ 환경 설정 완료")


---
## 📦 실습 데이터 + Weekend 2 사전 빌드

이 노트북은 Weekend 2의 결과물을 입력으로 가정합니다. Weekend 2를 끝내지 않았어도 아래 셀을 실행하면 자동으로 빌드됩니다:

| 객체 | 설명 |
|------|------|
| `mm_docs` | PDF 표/이미지/페이지 OCR → 멀티모달 Document 리스트 |
| `vs` | FAISS 인덱스 (`OpenAIEmbeddings`) |
| `weighted_search(...)` | `element_type` 가중치 검색 |
| `multimodal_rag_answer(...)` | 멀티모달 RAG 답변 (Graph-RAG의 baseline) |

> Weekend 2 노트북에서 더 큰 인덱스(오디오 + 비디오 포함)를 만들었다면 해당 인덱스를 사용해도 됩니다.


In [ ]:
# === Weekend 2 결과 재현: build_multimodal_documents ===
def _b64_image_message(path, prompt):
    raw = Path(path).read_bytes()
    b64 = base64.b64encode(raw).decode()
    ext = Path(path).suffix.lstrip(".").lower()
    mime = "image/jpeg" if ext in ("jpg", "jpeg") else f"image/{ext}"
    return HumanMessage(content=[
        {"type": "text", "text": prompt},
        {"type": "image_url", "image_url": {"url": f"data:{mime};base64,{b64}"}},
    ])


def _extract_tables_md(pdf_path, src):
    doc = fitz.open(pdf_path)
    out = []
    for pi, page in enumerate(doc, start=1):
        try:
            tabs = page.find_tables()
        except Exception:
            continue
        for tb in tabs.tables:
            try:
                df = tb.to_pandas().dropna(how="all").dropna(axis=1, how="all")
            except Exception:
                continue
            if df.shape[0] < 1 or df.shape[1] < 2:
                continue
            try:
                md_text = df.to_markdown(index=False)
            except Exception:
                md_text = "\n".join(["| " + " | ".join(str(v) for v in row) + " |" for row in df.values.tolist()])
            out.append(Document(page_content=f"**Table p{pi}**\n\n{md_text}",
                                metadata={"source": src, "page_number": pi, "element_type": "table"}))
    doc.close()
    return out


def _extract_image_captions(pdf_path, img_dir, src, max_images=2):
    doc = fitz.open(pdf_path)
    docs = []
    saved = 0
    for pi, page in enumerate(doc, start=1):
        for ii, info in enumerate(page.get_images(full=True), start=1):
            if saved >= max_images:
                break
            data = doc.extract_image(info[0])
            ext = data["ext"]
            p = Path(img_dir) / f"{Path(pdf_path).stem}_p{pi}_img{ii}.{ext}"
            p.write_bytes(data["image"])
            cap = vision.invoke([_b64_image_message(str(p), "이 이미지를 한국어로 1~2문장으로 설명하세요.")]).content
            docs.append(Document(page_content=cap,
                                 metadata={"source": src, "page_number": pi, "element_type": "image_caption"}))
            saved += 1
        if saved >= max_images:
            break
    doc.close()
    return docs


def _page_ocr(pdf_path, img_dir, src, max_pages=1):
    docs = []
    doc = fitz.open(pdf_path)
    stem = Path(pdf_path).stem
    for pi, page in enumerate(doc, start=1):
        if pi > max_pages:
            break
        pix = page.get_pixmap(dpi=120)
        p = Path(img_dir) / f"{stem}_p{pi}.png"
        pix.save(str(p))
        ocr = vision.invoke([_b64_image_message(str(p),
            "이 이미지의 모든 텍스트를 원본 순서대로 추출하세요. 표는 행/열을 보존하세요. 해석 금지.")]).content
        docs.append(Document(page_content=ocr,
                             metadata={"source": src, "page_number": pi, "element_type": "page_ocr"}))
    doc.close()
    return docs


def build_multimodal_documents(pdf_path, img_dir):
    src = Path(pdf_path).name
    return (_extract_tables_md(pdf_path, src)
            + _extract_image_captions(pdf_path, img_dir, src, max_images=2)
            + _page_ocr(pdf_path, img_dir, src, max_pages=1))


mm_docs = build_multimodal_documents(str(PDF_REPORT), str(IMG_DIR))
print(f"멀티모달 Documents: {len(mm_docs)}")
print(Counter(d.metadata["element_type"] for d in mm_docs))


# === Weekend 2 결과 재현: FAISS 인덱스 + 가중치 검색 + 멀티모달 RAG ===
vs = FAISS.from_documents(mm_docs, embeddings)
print(f"\n✅ FAISS 인덱스 — {vs.index.ntotal} 벡터")


DEFAULT_WEIGHTS = {
    "table": 1.4, "page_ocr": 1.0, "image_caption": 0.8,
    "audio_chunk": 1.1, "video_frame_caption": 0.9, "video_audio_chunk": 1.1,
}


def weighted_search(vs, query, weights=None, k=5, fetch_k=20):
    w = DEFAULT_WEIGHTS if weights is None else weights
    hits = vs.similarity_search_with_score(query, k=fetch_k)
    scored = []
    for doc, distance in hits:
        et = doc.metadata.get("element_type", "")
        weight = w.get(et, 1.0)
        if weight == 0:
            continue
        scored.append((doc, weight / (distance + 1e-9)))
    scored.sort(key=lambda x: x[1], reverse=True)
    return scored[:k]


ELEMENT_LABELS = {
    "table": "표 (PDF)", "page_ocr": "페이지 텍스트", "image_caption": "이미지 캡션",
    "audio_chunk": "회의 발화", "video_frame_caption": "비디오 프레임", "video_audio_chunk": "비디오 발화",
}


def multimodal_rag_answer(vs, query, k=6):
    hits = weighted_search(vs, query, k=k)
    groups, sources = {}, []
    for doc, _ in hits:
        et = doc.metadata.get("element_type", "etc")
        groups.setdefault(et, []).append(doc.page_content)
        sources.append({"element_type": et, "snippet": doc.page_content[:80]})
    sections = []
    for et, contents in groups.items():
        body = "\n".join(f"- {c}" for c in contents)
        sections.append(f"### {ELEMENT_LABELS.get(et, et)}\n{body}")
    ctx = "\n\n".join(sections)
    sys = SystemMessage(content=(
        "당신은 멀티모달 RAG 어시스턴트. 컨텍스트의 각 섹션은 다른 모달리티에서 추출됨. "
        "근거 부족 시 '제공된 자료에서 확인 불가'라고 답하세요."
    ))
    user = HumanMessage(content=f"질문: {query}\n\n컨텍스트:\n{ctx}")
    return {"answer": llm.invoke([sys, user]).content, "sources": sources}


print("✅ weighted_search / multimodal_rag_answer 준비됨 (Graph-RAG의 baseline)")


---
## 문제 1: 멀티모달 Document에서 청크별 엔티티/관계 추출

`mm_docs`의 각 Document를 LLM에 보내 엔티티/관계를 JSON으로 추출. **modality 정보를 LLM에 알려주고**, 추출 결과 dict에 `element_type`을 보존합니다.

**요구사항:**
- 함수 시그니처: `extract_one_doc(doc: Document) -> dict`
- 프롬프트에 `element_type` 안내 ("이 텍스트는 표/이미지 캡션/오디오/비디오 ...에서 추출됨")
- 반환: `{"entities": [...], "relations": [...], "element_type": str, "source": str}`
- 파싱 실패 시 빈 entities/relations 반환

**평가기준:**
- 모든 mm_docs에 대해 호출 성공 (예외 없음)
- 첫 표(table) Document에서 적어도 1개 엔티티 추출
- 반환 dict에 4개 키 모두 존재


In [ ]:
import re

EXTRACT_PROMPT_TMPL = """다음 텍스트에서 엔티티와 관계를 JSON으로 추출하세요.
출처: {element_type}
스키마:
{{"entities": [{{"name": '...', "type": "Person|Organisation|Product|Concept|Metric"}}],
 "relations": [{{"source": '...', "target": '...', "label": '...'}}]}}
JSON만 출력하세요. 다른 텍스트 금지."""


def strip_fence(t):
    t = t.strip()
    if t.startswith("```"):
        t = re.sub(r"^```(?:json)?\s*", "", t); t = re.sub(r"\s*```$", "", t)
    return t


def extract_one_doc(doc: Document) -> dict:
    """단일 멀티모달 Document에서 엔티티/관계 추출."""
    # ---- 여기에 코드 작성 ----
    # 1) prompt = EXTRACT_PROMPT_TMPL.format(element_type=...)
    # 2) SystemMessage + HumanMessage(doc.page_content[:3000])
    # 3) llm.invoke → strip_fence → json.loads (실패 시 빈 결과)
    # 4) {"entities", "relations", "element_type", "source"} 반환
    return {"entities": [], "relations": [], "element_type": "", "source": ""}


# 테스트 — 표 Document 1개
table_doc = next((d for d in mm_docs if d.metadata.get('element_type') == "table"), mm_docs[0])
r = extract_one_doc(table_doc)
print(f"[{r['element_type']}] from {r['source']}")
print(f"  entities: {len(r['entities'])}, relations: {len(r['relations'])}")
for e in r['entities'][:3]:
    print(f"  • {e}")


---
## 문제 2: 청크 결과 병합 → 멀티모달 KG

청크별로 따로 추출한 결과를 하나의 `nx.DiGraph`로 병합. **modality 정보를 노드/엣지 속성으로 보존**해 나중에 Local Search에서 modality filter가 가능하게.

**요구사항:**
- 함수 시그니처: `merge_multimodal_graph(extraction_results: list[dict]) -> nx.DiGraph`
- 노드 속성: `type`, `source_doc_indices` (list[int]), `source_modalities` (list[str], 중복 X)
- 엣지 속성: `relations` (list[str], 중복 X), `source_modalities` (list[str], 중복 X)

**평가기준:**
- 여러 청크에 등장하는 엔티티가 *하나의 노드*로 병합
- 적어도 한 노드의 `source_modalities` 길이 >= 2 (여러 modality에 등장하는 엔티티)


In [ ]:
def merge_multimodal_graph(extraction_results: list[dict]) -> nx.DiGraph:
    """청크별 결과 → 단일 KG (modality 메타 보존)."""
    G = nx.DiGraph()
    # ---- 여기에 코드 작성 ----
    # 1) extraction_results 순회
    # 2) 각 entity: 이미 있으면 source_doc_indices/modalities에 append, 없으면 add_node
    # 3) 각 relation: 양 끝 노드 보장 + edge 누적 (relations list, source_modalities list)
    return G


G = merge_multimodal_graph(extraction_results)
print(f"멀티모달 KG — 노드 {G.number_of_nodes()}, 엣지 {G.number_of_edges()}")
# 여러 modality에 등장하는 엔티티 = 모달리티 가교 역할
multi_mod = [(n, d) for n, d in G.nodes(data=True) if len(d.get("source_modalities", [])) >= 2]
print(f"\n여러 모달리티에 등장하는 엔티티 {len(multi_mod)}개 (상위 5개):")
for n, d in sorted(multi_mod, key=lambda x: -len(x[1]['source_modalities']))[:5]:
    print(f"  • {n} ({d.get('type', '?')}; in {d['source_modalities']})")


---
## 문제 3: Leiden 커뮤니티 탐지 + modality 분포

Leiden으로 KG를 *주제 그룹*으로 분할. 각 커뮤니티의 **modality 분포**를 같이 계산해 "이 커뮤니티는 어느 모달리티 정보 위주인가" 진단.

**요구사항:**
- 함수 시그니처: `detect_communities(G: nx.DiGraph) -> dict[int, dict]`
- 반환: `{cid: {"nodes": [name], "modality_counts": {mod: count}, "size": int}}`
- `graspologic.partition.leiden(G.to_undirected())` 사용

**평가기준:**
- 커뮤니티 1개 이상
- 각 커뮤니티 dict에 `nodes`, `modality_counts`, `size` 키 존재


In [ ]:
from graspologic.partition import leiden


def detect_communities(G: nx.DiGraph) -> dict:
    """Leiden + modality 분포."""
    # ---- 여기에 코드 작성 ----
    # 1) parts = leiden(G.to_undirected())
    # 2) 커뮤니티별 노드 모으기
    # 3) 각 노드의 source_modalities를 Counter로 합산
    return {}


communities = detect_communities(G)
print(f"커뮤니티 {len(communities)}개:")
for cid in sorted(communities):
    info = communities[cid]
    mods = ", ".join(f"{m}:{c}" for m, c in sorted(info['modality_counts'].items(), key=lambda x: -x[1]))
    print(f"  C{cid} ({info['size']}노드)  [{mods}]")
    print(f"    nodes: {info['nodes'][:5]}{'...' if info['size']>5 else ''}")


---
## 문제 4: LLM 커뮤니티 요약 (modality 정보 포함)

각 커뮤니티에 대해 LLM에게 *엔티티 + 관계 + modality 분포*를 주고 한 단락 요약을 받습니다. modality 정보를 포함해야 LLM이 "이 커뮤니티는 표/회의 발화에서 주로 나온 거" 같은 *증거 출처*를 답에 반영.

**요구사항:**
- 함수 시그니처: `summarize_multimodal_community(G: nx.DiGraph, community_info: dict) -> str`
- 프롬프트에 modality_counts, 엔티티 목록(이름 + 출처 모달리티), 내부 관계 포함
- 4-6 문장 한국어 요약

**평가기준:**
- 빈 문자열 아님
- 가능하면 modality 단어("표", "회의", "비디오" 등)가 1회 이상 등장 (LLM 의존)


In [ ]:
COMMUNITY_PROMPT = """다음은 멀티모달 지식 그래프의 한 커뮤니티입니다.
모달리티 분포(어느 데이터 출처에서 왔는지): {modality_counts}

엔티티:
{entities}

내부 관계:
{relations}

이 커뮤니티의 *주제*와 *주요 사실*을 4-6 문장의 한국어로 요약하세요.
가능하면 어느 모달리티에서 나온 정보인지 짧게 언급해 주세요."""


def summarize_multimodal_community(G: nx.DiGraph, community_info: dict) -> str:
    """단일 커뮤니티 요약."""
    # ---- 여기에 코드 작성 ----
    # 1) entities 줄 만들기: "- name (type; from: [mod1, mod2])"
    # 2) 내부 엣지만 추출
    # 3) COMMUNITY_PROMPT.format(...) → llm.invoke
    return ""


# 첫 커뮤니티 시연
first_cid = sorted(communities.keys())[0]
print(f"=== C{first_cid} 요약 ===")
print(summarize_multimodal_community(G, communities[first_cid]))


---
## 문제 5: Multimodal Local Search — modality filter 지원

질문에서 엔티티를 매칭 → 이웃 → 원본 mm_docs 컨텍스트로 답변. **modality_filter** 옵션으로 "표만 보고 답하라" 같은 제약 가능.

**요구사항:**
- 함수 시그니처:
  ```python
  multimodal_local_search(question: str, G: nx.DiGraph, mm_docs: list[Document],
                          hops: int = 1, modality_filter: list[str] | None = None) -> str
  ```
- 동작:
  1. G의 노드 중 이름이 question에 포함된 것 찾기
  2. 매칭 노드의 `hops` 홉 이웃 확장
  3. 이웃 노드의 `source_doc_indices` 모으기
  4. `modality_filter`가 주어지면 `mm_docs[i].metadata['element_type']`이 그 안에 있는 것만
  5. 청크 텍스트 + 내부 엣지 정보를 컨텍스트로 LLM 호출

**평가기준:**
- 매칭 엔티티 없으면 "(매칭 없음)" 같은 안내 반환
- `modality_filter=['table']`로 호출 시 컨텍스트에 audio_chunk가 안 들어감


In [ ]:
def multimodal_local_search(question: str, G: nx.DiGraph,
                            mm_docs: list, hops: int = 1,
                            modality_filter: list | None = None) -> str:
    """엔티티 매칭 → 이웃 → 청크 → LLM 답변. modality filter 지원."""
    # ---- 여기에 코드 작성 ----
    # 1) matched = [n for n in G.nodes if n in question]
    # 2) hops 홉 이웃 확장
    # 3) source_doc_indices 합집합 → modality_filter 적용
    # 4) 컨텍스트 + 엣지 정보 → llm.invoke
    return ""


# 테스트 — 표만 보고 답하기
ans = multimodal_local_search(
    "Modu Tech의 Q4 매출은 얼마인가요?",
    G, mm_docs, hops=1, modality_filter=["table", "page_ocr"],
)
print(f"💬 (table+page_ocr only)\n{ans}")


---
## 문제 6: Multimodal Global Search — 커뮤니티 요약 종합

추상적/요약형 질문은 *모든 커뮤니티 요약*을 종합해 답.

**요구사항:**
- 함수 시그니처: `multimodal_global_search(question: str, community_summaries: dict, communities: dict) -> str`
- 프롬프트에 각 커뮤니티의 *modality 분포*도 같이 보여줌 → LLM이 "표 위주 커뮤니티 / 회의 위주 커뮤니티" 구별 가능
- 4-8 문장 한국어 답변

**평가기준:**
- 답변 길이 >= 50자


In [ ]:
def multimodal_global_search(question: str, community_summaries: dict, communities: dict) -> str:
    """모든 커뮤니티 요약을 종합해 답."""
    # ---- 여기에 코드 작성 ----
    # 1) 각 커뮤니티: "[C{cid} | modality: {counts}]\n{summary}"
    # 2) SystemMessage + HumanMessage
    # 3) llm.invoke
    return ""


print(multimodal_global_search(
    "이 문서/회의/영상에서 다루는 핵심 주제는 무엇이고, 모달리티별로 어떤 정보가 보강되었나요?",
    community_summaries, communities,
))


---
## 문제 7: Multimodal Graph-RAG — auto 라우팅 + baseline 결합

질문 → *엔티티 매칭 여부*로 Local/Global 자동 선택. Local 결과 + baseline FAISS 결과를 *함께* 사용해 더 풍부한 답변.

**요구사항:**
- 함수 시그니처:
  ```python
  multimodal_graphrag_answer(question: str, vs, G, mm_docs, community_summaries, communities,
                              hops: int = 1, modality_filter: list | None = None) -> dict
  ```
- 동작:
  1. G의 노드 중 question에 매칭되는 것 있나?
     - 있으면 Local + baseline FAISS top-3 결합
     - 없으면 Global
  2. 반환: `{"answer", "mode": "local"|"global", "matched_entities": [...]}`

**평가기준:**
- 엔티티가 있는 질문(예: "Modu Tech ...")은 mode="local"
- 추상적 질문(예: "전반적 트렌드")은 mode="global"


In [ ]:
def multimodal_graphrag_answer(
    question: str, vs, G: nx.DiGraph, mm_docs: list,
    community_summaries: dict, communities: dict,
    hops: int = 1, modality_filter: list | None = None,
) -> dict:
    """auto 라우팅 multimodal Graph-RAG."""
    # ---- 여기에 코드 작성 ----
    # 1) matched 확인
    # 2) Local: local_search + baseline FAISS top-3 청크 → 합쳐서 LLM
    # 3) Global: multimodal_global_search
    # 4) 반환 dict
    return {"answer": "", "mode": "", "matched_entities": []}


for q in [
    "Modu Tech의 Q4 매출과 회의에서 언급된 전략은?",
    "이 데이터의 전반적인 트렌드는?",
]:
    r = multimodal_graphrag_answer(q, vs, G, mm_docs, community_summaries, communities, hops=1)
    print(f"\n[{r['mode']}] 매칭: {r['matched_entities']}")
    print(f"  Q: {q}")
    print(f"  A: {r['answer'][:300]}...")


---
## 문제 8: query type에 따른 modality 가중치 자동 선택

질문 유형(정량/논의/시각)에 따라 modality 가중치가 다름. LLM으로 유형 분류 → 적합한 가중치 적용.

**요구사항:**
- Pydantic 모델: `class QueryType(BaseModel): type: Literal["quantitative", "discussion", "visual", "general"]`
- 함수 시그니처:
  ```python
  classify_and_answer(question: str, vs, G, mm_docs, community_summaries, communities) -> dict
  ```
- 동작:
  1. `llm.with_structured_output(QueryType)`로 분류
  2. type별 가중치 사전 사용:
     - quantitative → table/page_ocr 우대
     - discussion → audio_chunk/video_audio_chunk 우대
     - visual → image_caption/video_frame_caption 우대
     - general → 기본 가중치
  3. `multimodal_graphrag_answer(..., modality_filter=top_modalities)` 호출

**평가기준:**
- 반환 dict에 `query_type`, `answer`, `applied_modalities` 키


In [ ]:
from typing import Literal


class QueryType(BaseModel):
    type: Literal["quantitative", "discussion", "visual", "general"] = Field(
        ..., description="질문 유형 분류")
    reasoning: str = Field(..., description="짧은 분류 근거(한국어)")


MODALITY_PROFILES = {
    "quantitative": ["table", "page_ocr"],
    "discussion":   ["audio_chunk", "video_audio_chunk", "page_ocr"],
    "visual":       ["image_caption", "video_frame_caption", "page_ocr"],
    "general":      None,  # 모든 modality 허용
}


def classify_and_answer(question: str, vs, G, mm_docs,
                        community_summaries, communities) -> dict:
    """query 분류 → modality 가중 답변."""
    # ---- 여기에 코드 작성 ----
    # 1) classifier = llm.with_structured_output(QueryType); verdict = classifier.invoke(...)
    # 2) modality_filter = MODALITY_PROFILES[verdict.type]
    # 3) multimodal_graphrag_answer(..., modality_filter=modality_filter)
    return {"query_type": "", "answer": "", "applied_modalities": None}


for q in [
    "Q4 영업이익이 얼마인가요?",                      # quantitative
    "회의에서 누가 어떤 발언을 했나요?",                # discussion
    "비디오에서 보여준 그래프 모양은?",                # visual
]:
    r = classify_and_answer(q, vs, G, mm_docs, community_summaries, communities)
    print(f"\n[{r['query_type']}] {q}")
    print(f"  적용 modality: {r['applied_modalities']}")
    print(f"  답변: {r['answer'][:200]}...")


---
## 문제 9: Graph-RAG vs baseline FAISS — LLM-as-Judge 비교

같은 질문에 대해 *baseline RAG*와 *Graph-RAG* 두 답변을 받아 LLM이 비교 평가. Graph-RAG가 실제로 더 나은지 정량 측정.

**요구사항:**
- Pydantic 모델: `class Comparison(BaseModel): better: Literal["baseline", "graphrag", "tie"]; reasoning: str`
- 함수 시그니처: `judge_comparison(question, vs, G, mm_docs, community_summaries, communities) -> dict`
- 두 답변 다 받고 → judge LLM 호출 → 어느 게 나은지 결정
- 반환: `{"baseline_answer", "graphrag_answer", "better", "reasoning"}`

**평가기준:**
- 반환 dict에 4개 키 모두 존재
- `better`는 3가지 값 중 하나


In [ ]:
class Comparison(BaseModel):
    better: Literal["baseline", "graphrag", "tie"] = Field(...)
    reasoning: str = Field(..., description="짧은 비교 근거(한국어)")


def judge_comparison(question: str, vs, G, mm_docs,
                     community_summaries, communities) -> dict:
    """두 RAG 답변 비교."""
    # ---- 여기에 코드 작성 ----
    # 1) baseline = multimodal_rag_answer(vs, question)['answer']
    # 2) graphrag = multimodal_graphrag_answer(question, vs, G, mm_docs, ..., communities)['answer']
    # 3) judge_llm = llm.with_structured_output(Comparison)
    # 4) "어느 답변이 더 충실하고 풍부한가?" 평가
    return {"baseline_answer": "", "graphrag_answer": "", "better": "", "reasoning": ""}


# 실패 케이스 후보: multi-hop, 모달리티 횡단
q = "Modu Tech 매출 추이를 표와 회의 발언을 종합해 설명해주세요"
r = judge_comparison(q, vs, G, mm_docs, community_summaries, communities)
print(f"❓ {q}")
print(f"\n[baseline]\n{r['baseline_answer'][:300]}...")
print(f"\n[graphrag]\n{r['graphrag_answer'][:300]}...")
print(f"\n⚖️  Better: {r['better']}\n   Reason: {r['reasoning']}")


---
## 문제 10: 최종 미션 — `MultimodalGraphRAGService` 통합 클래스 🏁

문제 1-9의 모든 단계를 단일 클래스로 캡슐화. 운영 시 한 객체로 인덱싱 + 쿼리 + 비교 모두.

**요구사항:**
- 클래스 시그니처:
  ```python
  class MultimodalGraphRAGService:
      def __init__(self, mm_docs: list[Document], vs: FAISS): ...
      def index(self) -> None: ...                                 # 1-4 단계
      def query(self, question: str, mode: str = "auto", modality_filter=None) -> dict: ...
      def stats(self) -> dict: ...                                 # 인덱싱 통계
      def compare(self, question: str) -> dict: ...                # 문제 9 재사용
  ```

- `mode`:
  - `"auto"` (기본) — 엔티티 매칭 → local, 없으면 global
  - `"local"` — 강제 local
  - `"global"` — 강제 global
  - `"adaptive"` — 문제 8의 classify_and_answer 사용

**평가기준:**
- `svc.index()` 호출 후 `svc.stats()` 결과에 노드/엣지/커뮤니티 수 포함
- `svc.query('...' mode="adaptive")` 동작
- `svc.compare('...')` 결과에 4개 키 (`baseline_answer`, `graphrag_answer`, `better`, `reasoning`)


In [ ]:
class MultimodalGraphRAGService:
    """멀티모달 Graph-RAG 통합 서비스."""

    def __init__(self, mm_docs: list, vs: FAISS):
        self.mm_docs = mm_docs
        self.vs = vs
        # index() 호출 후 채워짐
        self.extraction_results = None
        self.G = None
        self.communities = None
        self.summaries = None

    def index(self) -> None:
        """엔티티 추출 → 그래프 병합 → 커뮤니티 → 요약."""
        # ---- 여기에 코드 작성 ----
        pass

    def stats(self) -> dict:
        """인덱싱 통계."""
        # ---- 여기에 코드 작성 ----
        return {}

    def query(self, question: str, mode: str = "auto",
              modality_filter: list | None = None) -> dict:
        """쿼리 — mode에 따라 라우팅."""
        # ---- 여기에 코드 작성 ----
        return {}

    def compare(self, question: str) -> dict:
        """baseline vs Graph-RAG 비교."""
        # ---- 여기에 코드 작성 ----
        return {}


# 테스트
svc = MultimodalGraphRAGService(mm_docs, vs)
svc.index()
print("📊 stats:", svc.stats())

for q, mode in [
    ("Modu Tech Q4 매출은?", "auto"),
    ("회의에서 다룬 주요 결정은?", "adaptive"),
    ("코퍼스 전반 주제는?", "global"),
]:
    r = svc.query(q, mode=mode)
    print(f"\n[{r.get('mode', mode)}] {q}")
    print(f"  → {r.get('answer', '')[:200]}...")

cmp = svc.compare("매출 추이를 표와 회의 발언으로 설명")
print(f"\n⚖️  Comparison: {cmp['better']} — {cmp['reasoning']}")
